# ⚡ STAGE 1: CHUNKER & BM25s SPARSE INDEX BUILDER (CPU-ONLY)
### Road to AI 2026 (R2AI) — ViBioMIR Medical Document Retrieval

> **Tài nguyên**: Chạy trên **Kaggle CPU-Only (4 vCPUs, 30 GB RAM, 0 giờ GPU!)**.
> **Thời gian chạy**: ~12 – 18 phút cho toàn bộ 1.025M bài viết.
> **Mục tiêu**:
> 1. Tải 14 shards Parquet corpus (1,025,647 bài viết) từ HF Bucket.
> 2. Phân đoạn **Chunk lớn 1,500 ký tự** (overlap 250 ký tự) để giải quyết dứt điểm ngưỡng Token Overlap $\ge 0.40$ của BTC.
> 3. Lưu toàn bộ metadata vào SQLite `chunks_meta.sqlite`.
> 4. Xây dựng chỉ mục thưa đa ngôn ngữ siêu tốc **BM25s** (Mmap, zero-RAM).
> 5. Đồng bộ `chunks_meta.sqlite` và `bm25s_index/` lên Hugging Face Bucket để Stage 2 (GPU) sử dụng.

In [ ]:
# 1. Cài đặt các gói phụ thuộc siêu tốc với uv
!pip install -q uv
!uv pip install --system --no-cache pyarrow "huggingface_hub[cli]" bm25s pyvi jieba loguru tqdm

In [ ]:
# 2. Thiết lập mã nguồn dự án
import os
from pathlib import Path

REPO_URL = "https://github.com/nieths/med-doc-retrieval.git"
PROJECT_DIR = Path("/kaggle/working/med-doc-retrieval")

if not PROJECT_DIR.exists():
    print("🚀 Đang clone repository từ GitHub...")
    !git clone -b main $REPO_URL $PROJECT_DIR
else:
    print("🔄 Đang cập nhật repository...")
    %cd $PROJECT_DIR
    !git pull origin main

%cd $PROJECT_DIR
print("Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# 3. Cấu hình bảo mật Hugging Face Token
import os
from huggingface_hub import HfApi

# Ưu tiên lấy từ Kaggle Secrets (nếu đã thêm secret 'HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Đã lấy HF_TOKEN an toàn từ Kaggle Secrets!")
except Exception:
    # Fallback dự phòng nếu chưa cài Kaggle Secret
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if not HF_TOKEN:
        print("💡 Hãy thêm secret 'HF_TOKEN' vào Kaggle Secrets hoặc nhập token của bạn.")

os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

In [ ]:
# 4. Tải 14 shards Parquet Corpus (1.025M bài viết) từ Hugging Face Bucket
import os
from pathlib import Path

corpus_dir = Path("data/processed/parquet_corpus")
corpus_dir.mkdir(parents=True, exist_ok=True)

# Danh sách 14 shards từ corpus_part_000 đến corpus_part_013
corpus_files = [(f"corpus/corpus_part_{i:03d}.parquet", f"data/processed/parquet_corpus/corpus_part_{i:03d}.parquet") for i in range(14)]

missing_shards = []
for remote, local in corpus_files:
    lp = Path(local)
    if not lp.exists() or lp.stat().st_size == 0:
        missing_shards.append((remote, local))
    else:
        print(f"✓ Đã có: {lp.name} ({lp.stat().st_size / (1024*1024):.2f} MB)")

if missing_shards:
    print(f"\nĐang tải {len(missing_shards)} shards còn thiếu từ Hugging Face Bucket...")
    api.download_bucket_files(
        bucket_id="nieths/ViBioMIR",
        files=missing_shards,
        token=HF_TOKEN,
    )
    print("✅ Tải xong toàn bộ 14 shards corpus!")
else:
    print("\n✅ Đã có đầy đủ toàn bộ 14 shards corpus!")

In [ ]:
# 5. Phân đoạn văn bản (Chunk size = 1,500 ký tự) & Ghi vào SQLite
import sqlite3
import time
from tqdm import tqdm
import pyarrow.parquet as pq
from src.ingestion.chunker import DocumentChunker
from src.ingestion.cleaner import normalize_text, detect_language

indices_dir = Path("data/indices")
indices_dir.mkdir(parents=True, exist_ok=True)
sqlite_path = indices_dir / "chunks_meta.sqlite"

# Thiết lập Chunker tối ưu cho tiêu chuẩn BTC
chunker = DocumentChunker(
    max_chunk_size=1500,     # Chuẩn độ dài gold paragraph của BTC (~300-350 tokens)
    chunk_overlap=250,       # Đảm bảo giữ trọn ngữ cảnh y khoa giữa các câu
    min_chunk_size=80,       # Lọc bỏ rác/mẩu câu vụn
    split_by_sentences=True, # Cắt chính xác tại biên câu
    enable_contextual=True,  # Giữ tiêu đề bài viết
)

print(f"Khởi tạo cơ sở dữ liệu SQLite tại: {sqlite_path}...")
if sqlite_path.exists():
    sqlite_path.unlink()  # Làm mới hoàn toàn

conn = sqlite3.connect(sqlite_path)
cur = conn.cursor()
cur.execute("PRAGMA synchronous = OFF")
cur.execute("PRAGMA journal_mode = MEMORY")
cur.execute("""
CREATE TABLE chunks (
    chunk_id INTEGER PRIMARY KEY,
    doc_id BIGINT,
    title TEXT,
    chunk_text TEXT,
    contextual_text TEXT,
    url TEXT,
    domain TEXT,
    lang TEXT,
    char_count INTEGER
)
""")

cur.execute("CREATE INDEX idx_chunks_doc_id ON chunks (doc_id)")
cur.execute("CREATE INDEX idx_chunks_domain ON chunks (domain)")

global_chunk_id = 0
insert_buffer = []
t0 = time.time()

corpus_shards = sorted(corpus_dir.glob("corpus_part_*.parquet"))
print(f"Bắt đầu cắt chunk cho {len(corpus_shards)} shards...")

for shard_file in tqdm(corpus_shards, desc="Processing Shards"):
    tbl = pq.read_table(shard_file)
    doc_ids = tbl["doc_id"].to_pylist()
    urls = tbl["url"].to_pylist()
    titles = tbl["title"].to_pylist()
    texts = tbl["text"].to_pylist()
    langs = tbl["lang"].to_pylist()
    domains = tbl["domain"].to_pylist()
    
    for did, u, tit, txt, lg, dom in zip(doc_ids, urls, titles, texts, langs, domains):
        if not txt or len(txt.strip()) < 50:
            continue
            
        doc_chunks = chunker.chunk_document(
            doc_id=str(did),
            text=txt,
            title=tit or "",
            lang=lg or "vi",
        )
        
        for ch in doc_chunks:
            insert_buffer.append((
                global_chunk_id,
                int(did),
                tit or "",
                ch.chunk_text,
                ch.contextual_text or ch.chunk_text,
                u or "",
                dom or "",
                ch.lang,
                len(ch.chunk_text),
            ))
            global_chunk_id += 1
            
            if len(insert_buffer) >= 20000:
                cur.executemany("""
                INSERT INTO chunks (chunk_id, doc_id, title, chunk_text, contextual_text, url, domain, lang, char_count)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, insert_buffer)
                conn.commit()
                insert_buffer = []

if insert_buffer:
    cur.executemany("""
    INSERT INTO chunks (chunk_id, doc_id, title, chunk_text, contextual_text, url, domain, lang, char_count)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, insert_buffer)
    conn.commit()
    insert_buffer = []

conn.close()
elapsed = time.time() - t0
size_mb = sqlite_path.stat().st_size / (1024 * 1024)
print("=" * 60)
print(f"🎉 HOÀN TẤT CHUNKING VÀ TẠO METADATA SQLITE:")
print(f"  Tổng số chunks tạo ra: {global_chunk_id:,}")
print(f"  Dung lượng SQLite:     {size_mb:.2f} MB")
print(f"  Thời gian thực hiện:   {elapsed/60:.2f} phút")
print("=" * 60)

In [ ]:
# 6. Xây dựng chỉ mục thưa BM25s (Zero-RAM Streaming Iterator)
import json
import re
import sqlite3
import time
import bm25s

bm25s_dir = indices_dir / "bm25s_index"
bm25s_dir.mkdir(parents=True, exist_ok=True)
RE_WORDS = re.compile(r"\w+")

class SqliteStreamingTokens:
    """Stream từng chunk từ SQLite và tokenize on-the-fly, không ngốn RAM."""
    def __init__(self, db_path):
        self.db_path = str(db_path)

    def __iter__(self):
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        for row in cur.execute("SELECT contextual_text, lang FROM chunks ORDER BY chunk_id"):
            text = (row[0] or "").lower()
            lang = row[1] or "vi"
            
            # Tokenizer tiếng Trung (unigram + bigram)
            if lang == "zh" or any("\u4e00" <= c <= "\u9fff" for c in text[:60]):
                chars = [c for c in text if not c.isspace()]
                yield chars + [chars[i] + chars[i + 1] for i in range(len(chars) - 1)]
            else:
                # Tiếng Việt và Tiếng Anh (Word-level regex)
                yield RE_WORDS.findall(text)
        conn.close()

print("🚀 Khởi tạo mô hình BM25s (k1=1.5, b=0.75)...")
retriever = bm25s.BM25(k1=1.5, b=0.75, method="robertson")

t0 = time.time()
corpus_tokens = SqliteStreamingTokens(sqlite_path)

print("Đang index dữ liệu vào BM25s (tốc độ ~35k chunks/giây)...")
retriever.index(corpus_tokens, show_progress=True)

print(f"Lưu BM25s index vào: {bm25s_dir}...")
retriever.save(str(bm25s_dir), corpus=None)

# Lưu metadata phụ trợ
meta_path = indices_dir / "bm25s_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump({
        "num_chunks": global_chunk_id,
        "max_chunk_size": 1500,
        "chunk_overlap": 250,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, f, indent=2)

elapsed = time.time() - t0
print("=" * 60)
print(f"🎉 HOÀN TẤT BUILD BM25s SPARSE INDEX:")
print(f"  Thời gian index: {elapsed/60:.2f} phút")
print("=" * 60)

In [ ]:
# 7. Kiểm tra truy hồi thử nghiệm trên BM25s vừa tạo
test_queries = [
    "sốt xuất huyết dùng thuốc gì",
    "hepatitis B transmission prevention",
    "糖尿病的早期症状有哪些"
]

print("Thử nghiệm search trên 3 ngôn ngữ:")
for q in test_queries:
    q_tokens = [q.lower().split()]
    docs, scores = retriever.retrieve(q_tokens, k=3)
    print(f"\nQuery: '{q}'")
    print(f"  Top 1 chunk_id: {docs[0, 0]} (Score: {scores[0, 0]:.4f})")

In [ ]:
# 8. Đồng bộ chunks_meta.sqlite và bm25s_index lên Hugging Face Storage Bucket
import subprocess
import shutil

hf_bin = shutil.which("hf") or "hf"
env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN

print("📤 Đang upload chunks_meta.sqlite lên HF Bucket...")
!hf buckets cp data/indices/chunks_meta.sqlite hf://buckets/nieths/ViBioMIR/indices/chunks_meta.sqlite

print("📤 Đang upload bm25s_metadata.json lên HF Bucket...")
!hf buckets cp data/indices/bm25s_metadata.json hf://buckets/nieths/ViBioMIR/indices/bm25s_metadata.json

print("📤 Đang sync thư mục bm25s_index/ lên HF Bucket...")
!hf buckets sync data/indices/bm25s_index hf://buckets/nieths/ViBioMIR/indices/bm25s_index

print("\n" + "=" * 60)
print("🎉 STAGE 1 HOÀN TẤT XUẤT SẮC!")
print("Toàn bộ metadata và chỉ mục BM25s mới đã sẵn sàng trên Hugging Face!")
print("Bây giờ bạn có thể mở Notebook 2 (GPU) để chạy Dense Embedding BGE-M3!")
print("=" * 60)